# StyleWeave Phase 1 B0 (Kaggle)

This notebook **imports repository code**. It does not reimplement ranking.

**Settings:** Internet on. Accelerator **GPU T4** (prefer T4 over P100: current Kaggle PyTorch CUDA builds may lack P100 kernels). This uses the free quota, not a paid GPU.

**Do not upload FashionIQ images.** CDLA variant is unspecified. Attach an *existing* input if you accept its provenance, then let coverage fail honestly if IDs are missing.

Optional existing image dataset (license on Kaggle: `Other`, description empty — **not a verified redistribution grant**):
`https://www.kaggle.com/datasets/hoangkim14/fashion-iq-dataset`

Official text-only Kaggle copy (7.36 MB, Kaggle license `Unknown`):
`https://www.kaggle.com/datasets/nguyenph/fashion-iq`

Hugging Face publishing stays **off**.

In [ ]:
import os, sys, subprocess, pathlib

REPO_DIR = pathlib.Path("/kaggle/working/styleweave-vlm-lab")
REPO_URL = "https://github.com/SumalyaPatnala/styleweave-vlm-lab.git"

if not REPO_DIR.exists():
    token = os.environ.get("GITHUB_TOKEN") or (open("/run/secrets/GITHUB_TOKEN").read().strip() if pathlib.Path("/run/secrets/GITHUB_TOKEN").exists() else "")
    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    subprocess.check_call(["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)])

sys.path.insert(0, str(REPO_DIR / "src"))
print("repo", REPO_DIR, "exists", REPO_DIR.exists())

In [ ]:
import glob, pathlib
from styleweave.data.fashioniq import discover_fashioniq_layout

candidates = [pathlib.Path(p) for p in glob.glob("/kaggle/input/*")]
layout = None
for root in candidates:
    layout = discover_fashioniq_layout(root)
    if layout:
        break
print(layout)

In [ ]:
from styleweave.config import load_b0_config
from styleweave.eval.b0 import run_b0, CoverageError

assert layout is not None, "Attach a FashionIQ layout (captions/ + image_splits/ + images/) before running."

cfg = load_b0_config(REPO_DIR / "configs/experiments/phase1_frozen_clip.yaml", root=REPO_DIR)
cfg.captions_dir = str(layout["captions_dir"])
cfg.image_splits_dir = str(layout["image_splits_dir"])
cfg.images_dir = str(layout["images_dir"])
cfg.artifacts_dir = "/kaggle/working/artifacts/phase1_frozen_clip"
cfg.allow_download = True  # CLIP weights on Kaggle disk, not the Mac
cfg.hub.enabled = False

coverage = run_b0(
    cfg,
    dry_run=True,
    images_source="kaggle-input:" + str(layout["root"]),
)
print("coverage_ok", coverage["coverage"]["ok"])
print(coverage["coverage"]["errors"])

In [ ]:
# Full B0. Skip if coverage failed. Does not publish to Hugging Face.
from pathlib import Path
import json

if not coverage["coverage"]["ok"]:
    raise SystemExit("Coverage failed; not encoding CLIP and not writing metrics.json")

payload = run_b0(
    cfg,
    dry_run=False,
    images_source="kaggle-input:" + str(layout["root"]),
    cache_dir="/kaggle/working/hf-cache",
)
metrics_path = Path(cfg.artifacts_dir) / "metrics.json"
print(metrics_path.read_text()[:2000])
print("hub_published", payload.get("hub_published"))